# 08 · Dates

**Dataset:** 1,000 synthetic healthcare claims (January–June 2026). All data is fake.

| Table | Columns |
|---|---|
| `claims` (1,000) | `claim_id`, `patient_id`, `provider_id`, `date_of_service`, `cpt_code`, `billed_amount`, `paid_amount`, `claim_status`, `denial_reason` |
| `patients` (200) | `patient_id`, `patient_name`, `date_of_birth`, `insurance_plan`, `state` |
| `providers` (13) | `provider_id`, `provider_name`, `specialty`, `state` |
| `procedures` (13) | `cpt_code`, `description`, `service_type`, `standard_charge` |

> 🇪🇸 **Para Josué:** este tema ya lo sabes (lo viste el 17/11 con los 10 claims): `DATE_TRUNC`, restar fechas y el *aging*. Aquí lo usas con 1,000 claims y aprendes **2 funciones nuevas chiquitas**: `DAYNAME` y `QUARTER`.
> Es el **último tema**: con estos 4 ejercicios llegas a **45 de 45**. 🎯
> La primera vez, corre todo con **Run All**. **Guarda con Ctrl + S después de cada ejercicio.**

In [1]:
# Load the CSV files as tables.
import duckdb

duckdb.sql("CREATE OR REPLACE TABLE claims     AS SELECT * FROM '../data/claims.csv'")
duckdb.sql("CREATE OR REPLACE TABLE patients   AS SELECT * FROM '../data/patients.csv'")
duckdb.sql("CREATE OR REPLACE TABLE providers  AS SELECT * FROM '../data/providers.csv'")
duckdb.sql("CREATE OR REPLACE TABLE procedures AS SELECT * FROM '../data/procedures.csv'")
print("Tables loaded: claims, patients, providers, procedures")

Tables loaded: claims, patients, providers, procedures


---
## Example · Claims per month

> 🇪🇸 **Te recuerdo:** `DATE_TRUNC('month', fecha)` lleva cada fecha al **día 1 de su mes** (2026-03-17 → 2026-03-01). Es la "carpeta" del mes: todos los claims de marzo caen en la misma carpeta y el `GROUP BY` los junta.
> En el `GROUP BY` se repite la función completa, no el alias.

In [2]:
duckdb.sql("""
    SELECT DATE_TRUNC('month', date_of_service) AS service_month,
           COUNT(*) AS total_claims
    FROM claims
    GROUP BY DATE_TRUNC('month', date_of_service)
    ORDER BY service_month
""").df()

,service_month,total_claims
0,2026-01-01,190
1,2026-02-01,154
2,2026-03-01,178
3,2026-04-01,169
4,2026-05-01,148
5,2026-06-01,161


## Exercise 1 · Denial rate per month

Show, for each month, the number of claims and the **denial rate** (one decimal), from January to June.

Columns: `service_month`, `total_claims`, `denial_rate`.

✔ **6 rows.** January · 190 · 16.3. The lowest rate is **March (12.4)** and the highest is **June (25.5)**.

> 🇪🇸 El ejemplo de arriba, más la columna de la tasa de rechazo. **Te recuerdo** la fórmula:
> ```sql
> ROUND(100.0 * SUM(CASE WHEN claim_status = 'Denied' THEN 1 ELSE 0 END) / COUNT(*), 1) AS denial_rate
> ```
> Fíjate en junio: 1 de cada 4 claims rechazado. En tu análisis final, eso es una alerta: algo cambió ese mes.

In [12]:
duckdb.sql("""
    SELECT DATE_TRUNC('month', date_of_service) AS service_month,
    COUNT(*) AS total_claims,
    ROUND(100.0*SUM(CASE WHEN claim_status = 'Denied' THEN 1 ELSE 0 END)/COUNT(*), 1) AS denial_rate
    FROM claims
    GROUP BY DATE_TRUNC('month', date_of_service)
    ORDER BY service_month
""").df()

,service_month,total_claims,denial_rate
0,2026-01-01,190,16.3
1,2026-02-01,154,14.9
2,2026-03-01,178,12.4
3,2026-04-01,169,21.3
4,2026-05-01,148,14.2
5,2026-06-01,161,25.5


---
## Example · The day of the week

> 🇪🇸 **Nuevo:** dos funciones para el día de la semana:
> - `DAYNAME(fecha)` → el **nombre** del día: `Monday`, `Tuesday`...
> - `ISODOW(fecha)` → el **número** del día: lunes = 1 ... domingo = 7.
>
> ¿Para qué sirven las dos? Si ordenas por `DAYNAME`, SQL ordena **alfabéticamente** (Friday, Monday, Saturday...). Para que salga de lunes a domingo, ordenas por el **número**.

In [4]:
duckdb.sql("""
    SELECT claim_id, date_of_service,
           DAYNAME(date_of_service) AS day_name,
           ISODOW(date_of_service) AS day_number
    FROM claims
    LIMIT 3
""").df()

,claim_id,date_of_service,day_name,day_number
0,CL00001,2026-01-01,Thursday,4
1,CL00002,2026-01-01,Thursday,4
2,CL00003,2026-01-01,Thursday,4


## Exercise 2 · Claims by day of the week

Show, for each day of the week, the number of claims and the total billed, **from Monday to Sunday**.

Columns: `day_name`, `total_claims`, `total_billed`.

✔ **7 rows.** Monday · 125 · $48,741 … Sunday · 138 · $59,568. The day with the most money is **Friday ($72,137)**.

> 🇪🇸 Dos pistas:
> - `GROUP BY` con **las dos** funciones: `DAYNAME(date_of_service)` y `ISODOW(date_of_service)`. (Si no pones `ISODOW` en el `GROUP BY`, no la puedes usar en el `ORDER BY`).
> - `ORDER BY ISODOW(date_of_service)`. `ISODOW` **no** va en el `SELECT`: el enunciado pide solo 3 columnas.
>
> **Ojo de analista:** en la vida real, una clínica atiende muy poco el fin de semana. Aquí sábado y domingo tienen tantos claims como un martes. Con datos reales, eso sería una **pregunta de calidad de datos** (¿la fecha es la del servicio o la de carga?). Con datos sintéticos, es solo cómo se generaron.

In [ ]:
duckdb.sql("""
    SELECT 
    DAYNAME(date_of_service) AS day_name,
    COUNT(*) AS total_claims,
    SUM(billed_amount) AS total_billed
    FROM claims
    GROUP BY DAYNAME(date_of_service), ISODOW(date_of_service)
    ORDER BY ISODOW(date_of_service)
""").df()

,total_claims,total_billed,day_name
0,125,48741.0,Monday
1,157,63968.0,Tuesday
2,146,61225.0,Wednesday
3,154,64563.0,Thursday
4,143,72137.0,Friday
5,137,54883.0,Saturday
6,138,59568.0,Sunday


---
## Example · The quarter

> 🇪🇸 **Nuevo:** `QUARTER(fecha)` da el **trimestre** (*quarter*): enero–marzo = 1, abril–junio = 2, julio–septiembre = 3, octubre–diciembre = 4.
> En EE. UU. los reportes de negocio se hacen mucho por trimestre: "Q1", "Q2".

In [6]:
duckdb.sql("""
    SELECT QUARTER(date_of_service) AS service_quarter,
           COUNT(*) AS total_claims
    FROM claims
    GROUP BY QUARTER(date_of_service)
    ORDER BY service_quarter
""").df()

,service_quarter,total_claims
0,1,522
1,2,478


## Exercise 3 · Q1 vs Q2

Compare the two quarters: number of claims, total billed and denial rate (one decimal).

Columns: `service_quarter`, `total_claims`, `total_billed`, `denial_rate`.

✔ **2 rows:** 1 · 522 · $230,553 · 14.6 / 2 · 478 · $194,532 · **20.5**.

> 🇪🇸 El ejemplo de arriba, más `SUM(billed_amount)` y la fórmula de la tasa de rechazo (la misma del ejercicio 1).
> **Para tu análisis final:** en Q2 hay **menos** claims y **más** rechazos. Se factura menos y se pierde más.

In [23]:
duckdb.sql("""
    SELECT QUARTER(date_of_service) AS service_quarter,
    COUNT(*) AS total_claims,
    SUM(billed_amount) AS total_billed,
    ROUND(100.0*SUM(CASE WHEN claim_status = 'Denied' THEN 1 ELSE 0 END)/ COUNT(*),1) AS denial_rate
    FROM claims
    GROUP BY QUARTER(date_of_service)
    ORDER BY service_quarter
""").df()

,service_quarter,total_claims,total_billed,denial_rate
0,1,522,230553.0,14.6
1,2,478,194532.0,20.5


---
## Challenge ⭐⭐⭐ · Aging of the Pending claims

It's **July 15, 2026**. The billing team wants to know how old the **Pending** claims are. Group them by how many days have passed since `date_of_service`: `0-30`, `31-60`, `61-90` and `90+`. Show how many claims and how much money is in each group.

Columns: `aging_bucket`, `pending_claims`, `pending_amount`.

✔ **4 rows:** 0-30 · 3 · $548 / 31-60 · 16 · $5,976 / 61-90 · 12 · $4,961 / 90+ · **32** · **$12,063**.

> 🇪🇸 **Te recuerdo** dos cosas del 17/11:
> - Restar dos fechas da **días**: `DATE '2026-07-15' - date_of_service`.
> - El `CASE WHEN` va por orden: el primer `WHEN` que se cumple gana.
> ```sql
> CASE
>     WHEN DATE '2026-07-15' - date_of_service <= 30 THEN '0-30'
>     WHEN DATE '2026-07-15' - date_of_service <= 60 THEN '31-60'
>     ...
> END
> ```
> Completa el `CASE`, filtra solo los `Pending` y agrupa por el `CASE` (completo, no el alias).
> **Pista para el orden:** si ordenas por `aging_bucket`, ¿sale 0-30, 31-60, 61-90, 90+? Pruébalo.
> **Para tu análisis final:** la mitad de los Pending (32 de 63) ya pasó los 90 días. Muchas aseguradoras tienen un **plazo límite** (*timely filing limit*): pasado ese plazo, el claim ya no se paga.

In [24]:
duckdb.sql("""
    SELECT 
    CASE 
        WHEN DATE '2026-07-15' - date_of_service <= 30 THEN '0-30'
        WHEN DATE '2026-07-15' - date_of_service <= 60 THEN '31-60'
        WHEN DATE '2026-07-15' - date_of_service <= 90 THEN '61-90'
        ELSE '90+' END AS aging_bucket,
    COUNT (*) AS pending_claims,
    SUM(billed_amount) AS pending_amount
    FROM claims
    WHERE claim_status = 'Pending' 
    GROUP BY
    CASE 
        WHEN DATE '2026-07-15' - date_of_service <= 30 THEN '0-30'
        WHEN DATE '2026-07-15' - date_of_service <= 60 THEN '31-60'
        WHEN DATE '2026-07-15' - date_of_service <= 90 THEN '61-90'
        ELSE '90+' END
    ORDER BY aging_bucket
    
""").df()

,aging_bucket,pending_claims,pending_amount
0,0-30,3,548.0
1,31-60,16,5976.0
2,61-90,12,4961.0
3,90+,32,12063.0


---
## Check your understanding

> 🇪🇸 Responde en el chat, con tus palabras:
> 1. En el ejercicio 2, ¿por qué no basta con `ORDER BY day_name`?
> 2. ¿Qué diferencia hay entre `DATE_TRUNC('month', date_of_service)` y `MONTH(date_of_service)` si tus datos tuvieran claims de 2025 y de 2026?